# Train a prior on the compressed AMASS

Trains the unconditional pose prior (`SO3PriorTrainer`) on the compressed dataset made by `scripts/preprocess/compress_amass.py`, with the setup of the pretrained prior `id_0900` of the paper (12 flow layers, lr 0.005, batch 2000). Only the data differs, so the result can be compared with it.

- **Nothing is overwritten.** A run is saved to `RUN_ROOT/id_<EXP_ID>/`, outside the repository. If that folder exists the run goes to `id_<EXP_ID>_0`, `_1`, ... (`prepare_saving_process`). The pretrained models in `experiments/pretrained_models` are never touched.
- The training itself is `scripts/train_prior.py` (`main(config)`), unchanged.
- The data settings are made for this notebook only (environment variables of the kernel); `config/paths.yaml` stays as it is.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch
from omegaconf import OmegaConf

from hp.core.prior_training import load_train_script, make_config
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS
from hp.utils.paths import REPO_ROOT

# ---- parameters ----------------------------------------------------------------------
BASE_CONFIG = REPO_ROOT / "experiments" / "pretrained_models" / "id_0900" / "config.yaml"
COMPRESSED_RAW = "~/data/amass_compressed"            # output of compress_amass.py
COMPRESSED_ROTATIONS = "~/data/AMASS_rotations_c10"   # rotation files made from it (created below)
FULL_ROTATIONS = "~/data/AMASS_rotations"             # rotation files of the FULL data: the test below runs on these
RUN_ROOT = "~/data/experiments/prior_c10"             # new runs are saved here
EXP_ID = 9001
MAX_ITERATIONS = 2_000           # about 3.3 s per iteration on an M5 Pro CPU (MPS is not faster).
#   2_000 = the same number of epochs (about 2.7) over the compressed data as the paper's 20_000 over the full data.
#   20_000 = the same number of iterations, 10x as many epochs, about 19 hours on this CPU.
VALID_INTERVAL = 500
VALID_WASSERSTEIN = False        # the Wasserstein distance on the validation set is slow
DEVICE = "cpu"                   # "cpu", "mps", "cuda:0" or "auto" (on a Mac cpu is as fast as mps here)
SEED = 10
CLIP_GRAD_NORM = 1.0             # not in the old config of id_0900; the value of config/config.yaml
SAVE_CKPT_EVERY = 1000           # same: a checkpoint of the model every n iterations
# ----------------------------------------------------------------------------------------

expand = lambda p: str(Path(p).expanduser())
assert Path(expand(RUN_ROOT)).resolve() != (REPO_ROOT / "experiments").resolve(), "RUN_ROOT must not be the tracked experiments folder"
assert "pretrained_models" not in expand(RUN_ROOT), "RUN_ROOT must not be inside pretrained_models"
assert expand(FULL_ROTATIONS) != expand(COMPRESSED_ROTATIONS), "FULL_ROTATIONS must be the folder of the full data"
assert all((Path(expand(FULL_ROTATIONS)) / f"{n}.pt").is_file() for n in ORIGINAL_AMASS_SPLITS["test"]), "the full test rotation files are missing"
print("full data rotations:      ", expand(FULL_ROTATIONS))
print("compressed rotations:     ", expand(COMPRESSED_ROTATIONS))
print("runs are saved below:     ", expand(RUN_ROOT))

## Data

The training reads `<Dataset>.pt` rotation files of the datasets in the official train/valid/test split, all three splits from the **compressed** folder. Missing ones are made from the compressed sequences with `preprocess_data.py` (a minute or two, only the first time). The files of the full data are not touched. The cell lists both: the compressed splits, and the full test data that the final comparison uses.

In [ ]:
needed = sorted({d for names in ORIGINAL_AMASS_SPLITS.values() for d in names})
os.makedirs(expand(COMPRESSED_ROTATIONS), exist_ok=True)
for name in needed:
    if not (Path(expand(COMPRESSED_ROTATIONS)) / f"{name}.pt").is_file():
        print("preprocessing", name, "...")
        subprocess.run(
            [sys.executable, str(REPO_ROOT / "scripts" / "preprocess" / "preprocess_data.py"), "--dataset", name,
             "--data_dir", expand(COMPRESSED_RAW), "--save_dir", expand(COMPRESSED_ROTATIONS)],
            check=True, capture_output=True,
        )
manifest = json.load(open(Path(expand(COMPRESSED_RAW)) / "compression.json"))
print(f"COMPRESSED data used for training and validation (and for the numbers main() prints): {expand(COMPRESSED_ROTATIONS)}")
print(f"compression: {manifest['params']}")
frames_by_split = {}
for split, names in ORIGINAL_AMASS_SPLITS.items():
    size = sum((Path(expand(COMPRESSED_ROTATIONS)) / f"{n}.pt").stat().st_size for n in names) / 1e9
    frames = sum(manifest["datasets"][n.replace("EyesJapanDataset", "Eyes_Japan_Dataset")]["frames_after"] for n in names)
    frames_by_split[split] = frames
    print(f"  {split:6s} {frames:>10,} frames, {size:.2f} GB: {', '.join(names)}")
full_test = {n: Path(expand(FULL_ROTATIONS)) / f"{n}.pt" for n in ORIGINAL_AMASS_SPLITS["test"]}
n_full = sum(torch.load(p).shape[0] for p in full_test.values())
print(f"FULL test data (not compressed), used for the final comparison: {expand(FULL_ROTATIONS)}")
print(f"  test   {n_full:>10,} frames, {sum(p.stat().st_size for p in full_test.values()) / 1e9:.2f} GB: {', '.join(full_test)}")

## Configuration

In [ ]:
# data and device of this notebook only (the kernel's environment)
os.environ["HUPROSO_AMASS_ROTATIONS"] = expand(COMPRESSED_ROTATIONS)
os.environ["HUPROSO_AMASS_SPLIT"] = "original"
os.environ.pop("HUPROSO_SUBSET", None)
if DEVICE != "auto":
    os.environ["HUPROSO_DEVICE"] = DEVICE

cfg = make_config(BASE_CONFIG, EXP_ID, expand(RUN_ROOT), MAX_ITERATIONS, VALID_INTERVAL, VALID_WASSERSTEIN,
                  DEVICE, SEED, CLIP_GRAD_NORM, SAVE_CKPT_EVERY)
print(OmegaConf.to_yaml(cfg)[:700], "...")
epochs = MAX_ITERATIONS * cfg.batch_size / frames_by_split["train"]
print(f"{MAX_ITERATIONS:,} iterations x batch {cfg.batch_size} = {epochs:.1f} epochs over the compressed training data; "
      f"about {MAX_ITERATIONS * 3.3 / 3600:.1f} h on an M5 Pro CPU")

## Train

Prints the training loss and, every `VALID_INTERVAL` iterations, the validation loss (a validation pass takes about 1.5 minutes). `main` also saves the model, the loss plot, `eval.json` and the TensorBoard logs into the run folder.

Note: the validation loss and the final numbers printed by `main` (and stored in `eval.json`) use the **compressed** valid and test data. The comparison below uses the full test data.

In [ ]:
train_prior = load_train_script()
train_prior.main(cfg)
run_dir = cfg.save.path_dir  # main() put the real run folder here (id_<EXP_ID>, or with a suffix)
print("saved to", run_dir)

## Compare with the pretrained prior

Both priors are evaluated on the **full** (not compressed) test split (`FULL_ROTATIONS`): the average log-likelihood of `num_eval_samples` random test poses, repeated a few times, and the distance of generated poses to the data. Higher likelihood and lower Wasserstein are better. The cell checks that exactly the full test set was loaded and saves the result as `eval_full_test.json` in the run folder.

In [ ]:
import importlib
import json

import numpy as np
import torch

from hp.core.evaluator import PriorEvaluator


n_full_test = sum(torch.load(Path(expand(FULL_ROTATIONS)) / f"{n}.pt").shape[0] for n in ORIGINAL_AMASS_SPLITS["test"])


def evaluate(model_dir, repeats=3):
    c = OmegaConf.load(Path(model_dir) / "config.yaml")
    c.only_eval, c.standard_mode, c.condition = True, True, False
    c.device = DEVICE
    c.save.path_dir = str(model_dir)
    np.random.seed(SEED), torch.manual_seed(SEED)
    trainer = getattr(importlib.import_module("hp.core.trainer"), c.trainer)(c)
    trainer.load_model(save_path=str(model_dir))
    n_test = len(trainer.eval_dataset)
    assert n_test == n_full_test, f"evaluated on {n_test} test frames, the full test set has {n_full_test}"
    ll = [trainer.eval()["eval_avg_ll"] for _ in range(repeats)]
    d = PriorEvaluator.eval(c, trainer, render_human=False, plot_samples=False, save_plot=False, compute_d_wasser=True)
    return dict(test_frames=n_test, ll=float(np.mean(ll)), ll_std=float(np.std(ll)), wasserstein=float(d["wasserstein_distance_ndim"]))


os.environ["HUPROSO_AMASS_ROTATIONS"] = expand(FULL_ROTATIONS)  # evaluate on the full test data
models = {"pretrained id_0900 (full data)": REPO_ROOT / "experiments" / "pretrained_models" / "id_0900",
          "trained on compressed data": Path(run_dir)}
try:
    results = {name: evaluate(path) for name, path in models.items()}
finally:
    os.environ["HUPROSO_AMASS_ROTATIONS"] = expand(COMPRESSED_ROTATIONS)
for name, r in results.items():
    print(f"{name:34s} {r['test_frames']:,} test frames | log-likelihood {r['ll']:8.2f} +- {r['ll_std']:.2f} | Wasserstein {r['wasserstein']:.3f}")
with open(Path(run_dir) / "eval_full_test.json", "w") as f:
    json.dump(results, f, indent=2)